# SOXX N-PORT 原始 ISIN 身份门\n\n假设：占位 CUSIP 的持仓仍可用同份 SEC 原件中的 ISIN 唯一识别。仅审计原始源字段；不读取市场收益、拟合模型或运行账户。

In [1]:
import collections, glob, hashlib, json, pathlib, xml.etree.ElementTree as ET
roots = [
 'data/snapshots/stage1_soxx_2019_nport_source_specimen_20261023/*.xml',
 'data/snapshots/stage1_soxx_2019_2022_nport_original_xml_20261023/*.xml',
 'data/snapshots/stage1_soxx_2023_2026_nport_original_source_20261023/*.xml',
]
files = sorted(p for pattern in roots for p in glob.glob(pattern))
rows = []
for path in files:
    report_rows = []
    for node in ET.parse(path).getroot().iter():
        if node.tag.rsplit('}', 1)[-1] != 'invstOrSec':
            continue
        fields = {x.tag.rsplit('}', 1)[-1]: x for x in node}
        value = lambda key: (fields[key].text or '').strip() if key in fields else ''
        if value('assetCat') != 'EC' or value('units') != 'NS':
            continue
        identifiers = fields.get('identifiers')
        isins = [] if identifiers is None else [x.attrib.get('value', '') for x in identifiers if x.tag.rsplit('}', 1)[-1] == 'isin']
        report_rows.append({'report': pathlib.Path(path).name, 'name': value('name'), 'cusip': value('cusip'), 'isin': isins[0] if len(isins) == 1 else '', 'lei': value('lei')})
    rows.extend(report_rows)
placeholders = [r for r in rows if r['cusip'] in ('000000000', 'N/A', '')]
collisions = []
for report in {r['report'] for r in rows}:
    ids = [r['isin'] for r in rows if r['report'] == report]
    if len(ids) != len(set(ids)):
        collisions.append(report)
nonplaceholder = collections.defaultdict(set)
for r in rows:
    if r['cusip'] not in ('000000000', 'N/A', ''):
        nonplaceholder[r['isin']].add(r['cusip'])
conflicts = {k: sorted(v) for k, v in nonplaceholder.items() if len(v) > 1}
result = {
 'original_xml_files': len(files),
 'original_xml_sha256': {pathlib.Path(p).name: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest() for p in files},
 'equity_shares_rows': len(rows),
 'placeholder_cusip_rows': len(placeholders),
 'placeholder_all_original_isin': all(bool(r['isin']) for r in placeholders),
 'placeholder_distinct_isins': len({r['isin'] for r in placeholders}),
 'placeholder_isin_issuer_examples': sorted({(r['name'], r['isin'], r['lei']) for r in placeholders}),
 'per_report_isin_collisions': collisions,
 'conflicting_nonplaceholder_cusip_per_isin': conflicts,
 'legacy_identity_qualified': False,
 'market_outcome_reads': 0, 'model_fits': 0, 'native_accounts': 0,
}
result['nport_security_identity_qualified'] = result['placeholder_all_original_isin'] and not collisions and not conflicts
print(json.dumps({k: v for k, v in result.items() if k != 'original_xml_sha256'}, ensure_ascii=False, indent=2))
pathlib.Path('research/stage1_soxx_nport_isin_identity_repair_20261023/result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n')


{
  "original_xml_files": 28,
  "equity_shares_rows": 840,
  "placeholder_cusip_rows": 71,
  "placeholder_all_original_isin": true,
  "placeholder_distinct_isins": 5,
  "placeholder_isin_issuer_examples": [
    [
      "ASML Holding NV",
      "USN070592100",
      "724500Y6DUVHQD6OXN27"
    ],
    [
      "Credo Technology Group Holding Ltd.",
      "KYG254571055",
      "N/A"
    ],
    [
      "Marvell Technology Group Ltd",
      "BMG5876H1051",
      "8DF36O58U3QIHUCGZB18"
    ],
    [
      "NXP Semiconductors NV",
      "NL0009538784",
      "724500M9BY5293JDF951"
    ],
    [
      "Nova Ltd.",
      "IL0010845571",
      "529900B2DSWE5V3SC292"
    ]
  ],
  "per_report_isin_collisions": [],
  "conflicting_nonplaceholder_cusip_per_isin": {},
  "legacy_identity_qualified": false,
  "market_outcome_reads": 0,
  "model_fits": 0,
  "native_accounts": 0,
  "nport_security_identity_qualified": true
}


3797

## 中文结论

28 份原始 N-PORT XML 共 840 条股票记录；71 条占位 CUSIP 均有原始 ISIN，涉及 5 个不同 ISIN。报告内无 ISIN 冲突，同一 ISIN 未映射冲突的有效 CUSIP。N-PORT 期间身份门通过。Credo 的 LEI 为 `N/A`，不能以 LEI 替代 ISIN。

运行范围仅是原始来源身份审计：零收益读取、零拟合、零账户。2011–2019 旧格式及交界仍有歧义，披露成分变化也不是实际调仓。未达到 Stage1 0bp 账户或 SOXX 超额目标，下一步先保守处理旧格式歧义，再冻结信息门。